# Notebook 03: Dimensión de Completitud

## Introducción

Comenzamos el estudio de las **6 Dimensiones de Calidad de Datos**. La primera y más fundamental es la **Completitud**.

### Objetivos:

1. Entender qué es completitud y por qué importa
2. Identificar datos faltantes
3. Validar completitud con Great Expectations
4. Generar reportes de completitud

## ¿Qué es Completitud?

**Completitud** mide si todos los valores requeridos están presentes en tus datos.

### Pregunta Clave:
> ¿Faltan datos donde no deberían faltar?

### Impacto de Negocio:

- **Análisis sesgado**: Datos faltantes pueden distorsionar estadísticas
- **Pérdida de ingresos**: No puedes contactar clientes sin email
- **Decisiones incorrectas**: Información incompleta → Conclusiones erróneas
- **Procesos rotos**: Pipelines fallan si faltan campos críticos

### Ejemplo Real:

En un e-commerce:
-  `customer_id` nulo → No puedes enviar el pedido
-  `email` nulo → No puedes enviar confirmación
-  `price` nulo → No puedes calcular ingresos
-  `discount_code` nulo → OK, es opcional

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np

print(f"Great Expectations versión: {gx.__version__}")

## Análisis Exploratorio de Completitud

Antes de validar, exploremos los datos para entender el problema.

In [ ]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Total de registros: {len(df)}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head(10)

In [ ]:
# Análisis de valores nulos
print("=" * 70)
print("ANÁLISIS DE COMPLETITUD")
print("=" * 70)

print("\n1. CONTEO DE VALORES NULOS POR COLUMNA:")
print("-" * 70)
nulos = df.isnull().sum()
for col, count in nulos.items():
    print(f"  {col:25s}: {count:4d} nulos")

print("\n2. PORCENTAJE DE COMPLETITUD POR COLUMNA:")
print("-" * 70)
completitud = (1 - df.isnull().sum() / len(df)) * 100
for col, pct in completitud.items():
    emoji = "" if pct == 100 else "⚠️" if pct >= 95 else ""
    print(f"  {emoji} {col:25s}: {pct:6.2f}%")

print("\n3. RESUMEN GENERAL:")
print("-" * 70)
total_valores = df.size
total_nulos = df.isnull().sum().sum()
pct_completitud_global = (1 - total_nulos / total_valores) * 100
print(f"  Total de valores: {total_valores:,}")
print(f"  Valores nulos: {total_nulos:,}")
print(f"  Completitud global: {pct_completitud_global:.2f}%")

## Clasificar Campos por Criticidad

No todos los campos tienen la misma importancia. Clasifiquemos:

### 🔴 Criticidad Alta (NO pueden ser nulos):
- `order_id`: Identificador único
- `customer_id`: Necesario para envío
- `price`: Necesario para facturación
- `quantity`: Necesario para inventario

### 🟡 Criticidad Media (Preferible que no sean nulos):
- `order_date`: Importante para análisis temporal
- `product_category`: Útil para segmentación

### 🟢 Criticidad Baja (Pueden ser nulos):
- `discount_code`: Opcional
- `notes`: Opcional

## Validar Completitud con Great Expectations

Ahora creemos expectativas para validar completitud.

In [ ]:
# Configurar contexto
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

print(" Contexto configurado")

In [ ]:
# Crear suite de completitud
suite = context.suites.add(gx.ExpectationSuite(name="completitud_ventas"))

# Campos de criticidad ALTA
campos_criticos = ["order_id", "customer_id", "price", "quantity"]

for campo in campos_criticos:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=campo,
            meta={
                "dimension": "Completitud",
                "criticidad": "Alta",
                "descripcion": f"{campo} es obligatorio para el negocio"
            }
        )
    )

# Campos de criticidad MEDIA (permitimos hasta 5% de nulos)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="order_date",
        mostly=0.95,  # Al menos 95% no nulos
        meta={
            "dimension": "Completitud",
            "criticidad": "Media",
            "descripcion": "Importante para análisis temporal"
        }
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product_category",
        mostly=0.95,
        meta={
            "dimension": "Completitud",
            "criticidad": "Media",
            "descripcion": "Útil para segmentación"
        }
    )
)

suite.save()
print(f"\n Suite creada con {len(suite.expectations)} expectativas de completitud")

In [ ]:
# Ejecutar validación
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite,
        name="validacion_completitud"
    )
)

resultado = validation_def.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("RESULTADO DE VALIDACIÓN DE COMPLETITUD")
print("=" * 70)
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")
print(f"\nExpectativas evaluadas: {len(resultado.results)}")
print(f"Expectativas exitosas: {sum(1 for r in resultado.results if r.success)}")
print(f"Expectativas fallidas: {sum(1 for r in resultado.results if not r.success)}")

# Detalles de fallas
print("\n" + "=" * 70)
print("DETALLES DE CAMPOS CON PROBLEMAS DE COMPLETITUD")
print("=" * 70)
for result in resultado.results:
    if not result.success:
        column = result.expectation_config.kwargs['column']
        unexpected_count = result.result.get('unexpected_count', 0)
        unexpected_percent = result.result.get('unexpected_percent', 0)
        criticidad = result.expectation_config.meta.get('criticidad', 'N/A')
        
        print(f"\n Campo: {column}")
        print(f"   Criticidad: {criticidad}")
        print(f"   Registros con nulos: {unexpected_count}")
        print(f"   Porcentaje de nulos: {unexpected_percent:.2f}%")

##  Ejercicio Práctico

Crea una función que genere un reporte de completitud para cualquier DataFrame.

### Requisitos:
1. Recibe un DataFrame
2. Calcula completitud por columna
3. Clasifica columnas por nivel de completitud:
   - Excelente: 100%
   - Bueno: 95-99%
   - Regular: 90-94%
   - Malo: < 90%
4. Retorna un DataFrame con el reporte

In [ ]:
def generar_reporte_completitud(df):
    """
    Genera un reporte de completitud para un DataFrame.
    
    Args:
        df: DataFrame de pandas
    
    Returns:
        DataFrame con reporte de completitud
    """
    # TU CÓDIGO AQUÍ
    pass

# Probar la función
# reporte = generar_reporte_completitud(df)
# print(reporte)

In [ ]:
# SOLUCIÓN (descomenta para ver)

# def generar_reporte_completitud(df):
#     reporte = pd.DataFrame({
#         'columna': df.columns,
#         'total_registros': len(df),
#         'valores_nulos': df.isnull().sum().values,
#         'valores_completos': df.notnull().sum().values,
#     })
#     
#     reporte['pct_completitud'] = (reporte['valores_completos'] / reporte['total_registros'] * 100).round(2)
#     
#     def clasificar(pct):
#         if pct == 100:
#             return 'Excelente'
#         elif pct >= 95:
#             return 'Bueno'
#         elif pct >= 90:
#             return 'Regular'
#         else:
#             return 'Malo'
#     
#     reporte['clasificacion'] = reporte['pct_completitud'].apply(clasificar)
#     
#     return reporte.sort_values('pct_completitud')

# # Probar
# reporte = generar_reporte_completitud(df)
# print("\nREPORTE DE COMPLETITUD:")
# print(reporte.to_string(index=False))

## Estrategias para Manejar Datos Faltantes

Cuando encuentras datos faltantes, tienes varias opciones:

### 1. Rechazar el Registro
- **Cuándo**: Campos críticos nulos
- **Ejemplo**: `customer_id` nulo → No puedes procesar el pedido

### 2. Imputar Valores
- **Cuándo**: Campos numéricos con pocos nulos
- **Métodos**: Media, mediana, moda, forward fill
- **Cuidado**: Puede introducir sesgos

### 3. Usar Valores por Defecto
- **Cuándo**: Campos opcionales
- **Ejemplo**: `discount_code` nulo → "NO_DISCOUNT"

### 4. Marcar como Desconocido
- **Cuándo**: No puedes imputar con confianza
- **Ejemplo**: `product_category` nulo → "UNKNOWN"

### 5. Investigar la Causa Raíz
- **Siempre**: Entender POR QUÉ faltan los datos
- **Ejemplo**: ¿Error en el sistema? ¿Cambio en el proceso?

## Generar Data Docs

Generemos un reporte visual de completitud.

In [ ]:
# Generar Data Docs
context.build_data_docs()

print("\n" + "=" * 70)
print(" Data Docs de Completitud generados!")
print("=" * 70)
print("\nEn el reporte verás:")
print("  - Expectativas de completitud por campo")
print("  - Porcentaje de valores nulos")
print("  - Clasificación por criticidad")
print("  - Gráficos de completitud")
print("\nAbriendo en tu navegador...")

context.open_data_docs()

##  Resumen del Notebook

Has aprendido sobre la dimensión de **Completitud**:

### Conceptos Clave:

1.  Completitud mide si todos los valores requeridos están presentes
2.  No todos los campos tienen la misma criticidad
3.  Usa `ExpectColumnValuesToNotBeNull` para validar
4.  El parámetro `mostly` permite tolerancia (ej: 95% completo)
5.  Clasifica campos por criticidad: Alta, Media, Baja

### Expectativas Aprendidas:

- `ExpectColumnValuesToNotBeNull`: Valida que no haya nulos
- Parámetro `mostly`: Permite un porcentaje de nulos
- Metadata: Documenta criticidad y razón de negocio

### Mejores Prácticas:

1. Clasifica campos por criticidad antes de validar
2. Usa `mostly` para campos de criticidad media
3. Documenta por qué cada campo es crítico
4. Investiga la causa raíz de datos faltantes
5. Define estrategias de manejo por tipo de campo

